# Generate factuality questions

In this notebook we query an LLM with the annotated dataset to generate

- Question to SmolDOC document
- Answer to question

and save it to disk, such that we do not have to ask the LLM multiple times (saving time and costs).

In [ ]:
from utils import get_extended_datasets

datasets = get_extended_datasets()

example_cfg = 'smoldoc__en_sw' # We use Swahili since this config has the full 584 document translations
annotated_dataset = datasets[example_cfg]
annotated_dataset

In [ ]:
import pandas as pd

# Get only the incorrect entries
df = pd.DataFrame(annotated_dataset)
incorrect_data = df[df["factuality"] == "has_errors"][:5] # Limit to first 5 examples
incorrect_data

In [ ]:
QUESTION_GEN_SYSTEM_PROMPT = """
You are an expert dataset creator for factuality evaluation.
Your task is to read an English document (from the SmolDoc dataset) and produce a small set of factual question, answer pairs that test a model's factual understanding of the text.

Your goals:
1. Create one or more (up to three) concise, factual, and self-contained questions based on the given document.
2. Each question must have one short, unambiguous gold answer that is explicitly supported by the text.
3. Questions should be neither trivial nor adversarial — they should test meaningful factual comprehension, not obscure details or wordplay.

Output only valid JSON, following this structure:

[
  {
    "question": "<English question>",
    "answer": "<short correct English answer>"
  }
]

- Limit each question to less than 25 words.
- Limit each answer to less than 10 words.
"""

In [ ]:
# id, question, answer

from llm_chat import CachedLLMChat, LLMChat, OllamaChatter, AzureOpenAIChatter

chatter = OllamaChatter(model_name="deepseek-r1:8b", think=False)
# chatter = AzureOpenAIChatter()
chat = LLMChat(chatter)
chat = CachedLLMChat(chat, cache_file_path="data/factuality_question_gen_cache.pkl")

questions_with_answers: list[dict[str, str]] = []

def parse_response(response: str, id: str):
    """
    Parse the LLM response as JSON and attach topic_id.

    Args:
        response (str): The LLM response string.
        id (str): The topic ID to attach.
    Returns:
        list[dict] | None: The parsed JSON with topic_id added, or None on failure.
    """
    try:
        import json
        json_data = response
        loaded_json = json.loads(json_data)
        for entry in loaded_json:
            entry["topic_id"] = id
        return loaded_json
    except Exception as e:
        print(f"Error parsing response for id {id}: {e}")
        print(f"Response was: {response}")
        return None


for idx, row in incorrect_data.iterrows():
    id = row["id"]
    srcs = " ".join(row["srcs"])
    errors = "\n\n".join((row["annotator_1_notes"], row["annotator_2_notes"], row["annotator_3_notes"]))
    chat.add_message("system", QUESTION_GEN_SYSTEM_PROMPT)
    response, thoughts = chat.chat(
        f"""Here is the source document: {srcs}\n
        Annotators have noted the following issues:
        {errors}\n
        Generate question and answers in JSON format."""
    )
    parsed_json = parse_response(response, id)
    questions_with_answers.extend(parsed_json or [])
    print(parsed_json)
    break

In [ ]:
df_questions = pd.DataFrame(questions_with_answers)
df_questions = df_questions[["topic_id", "question", "answer"]]
df_questions